# Week 7 Assignment

## Delta Lake MERGE Implementation

### Objective
The objective of this assignment is to demonstrate incremental data processing using Delta Lake. The workflow includes:
- Loading the dataset
- Performing data cleaning
- Creating incremental data
- Applying the MERGE operation
- Validating the results
- Displaying the final dataset

In [0]:
# ==========================================================
# Step 1: Import Required Libraries
# ==========================================================
# These libraries are used for data processing
# and Delta Lake MERGE operations.

from pyspark.sql.functions import *
from delta.tables import DeltaTable

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
# ==========================================================
# Step 2: Load the Superstore Dataset
# ==========================================================
# Read the Superstore table from the Databricks Catalog.

df = spark.table("workspace.default.sample_superstore")

# Display the dataset
display(df.limit(20))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


## Step 3: Explore the Dataset

In this step, we  will explore the dataset by checking the total number of rows, columns, schema, and summary info. This helps us to understand the structure before performing data cleaning.

In [0]:
# ==========================================================
# Step 3: Explore the Dataset
# ==========================================================
# Checking the total number of rows and columns

print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 9994
Total Columns: 21


In [0]:
# ==========================================================
# Displaying all the column names
# ==========================================================

print("Column Names:\n")

for column in df.columns:
    print(column)

Column Names:

Row ID
Order ID
Order Date
Ship Date
Ship Mode
Customer ID
Customer Name
Segment
Country
City
State
Postal Code
Region
Product ID
Category
Sub-Category
Product Name
Sales
Quantity
Discount
Profit


In [0]:
# ==========================================================
# Display the schema
# ==========================================================

df.printSchema()

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
# ==========================================================
# Display summary statistics
# ==========================================================

display(df.describe())

summary,Row ID,Order ID,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
count,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994
mean,4997.5,null,null,null,null,null,null,null,null,55190.3794276566,null,null,null,null,null,229.8580008304938,3.789573744246548,0.15620272163298934,28.656896307784802
stddev,2885.1636290974325,null,null,null,null,null,null,null,null,32063.693350365593,null,null,null,null,null,623.2451005086805,2.2251096911413994,0.2064519678257168,234.26010769095768
min,1,CA-2014-100006,First Class,AA-10315,Aaron Bergman,Consumer,United States,Aberdeen,Alabama,1040,Central,FUR-BO-10000112,Furniture,Accessories,"""While you Were Out"" Message Book, One Form per Page",0.444,1,0.0,-6599.978
max,9994,US-2017-169551,Standard Class,ZD-21925,Zuschuss Donatelli,Home Office,United States,Yuma,Wyoming,99301,West,TEC-PH-10004977,Technology,Tables,netTALK DUO VoIP Telephone Service,22638.48,14,0.8,8399.976


## Step 4: Data Cleaning

The dataset is cleaned by:
- Checking for null values
- Handling missing values
- Removing duplicate records



In [0]:
# ==========================================================
# Checking for the null values in each column
# ==========================================================

from pyspark.sql.functions import col, when, count

null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(null_counts)

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# ==========================================================
# Removing the duplicate rows
# ==========================================================

clean_df = df.dropDuplicates()

print("Rows before removing duplicates :", df.count())
print("Rows after removing duplicates  :", clean_df.count())

Rows before removing duplicates : 9994
Rows after removing duplicates  : 9994


## Analization
The dataset was analyzed for data quality before applying Delta Lake operations.

The following checks were performed:
- Checked for missing (null) values in all columns.
- Checked for duplicate records.

### Observation

- No missing values were found in any column.
- No duplicate records were found in the dataset.

Therefore, no additional cleaning operations were required, and the original dataset will use for further required operation.

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



## Step 5: Renaming Columns for Delta Compatibility

Delta Lake does not allow certain special characters (such as spaces) in column names by default.

To make the dataset compatible with Delta Lake, all column names are renamed by replacing spaces with underscores ( _ ).

In [0]:
# ==========================================================


for old_name in df.columns:
    new_name = old_name.replace(" ", "_")
    df = df.withColumnRenamed(old_name, new_name)

# Since there were no nulls or duplicates,
# this renamed DataFrame becomes our cleaned DataFrame.
clean_df = df

print("Updated Column Names:")
print(clean_df.columns)

Updated Column Names:
['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub-Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Step 5: Creating a Delta Table

In this step, the cleaned Superstore dataset is stored as a Delta table.

Delta Lake provides:
- ACID transactions
- Schema enforcement
- Efficient updates and deletes
- MERGE support for incremental data processing

In [0]:
# ==========================================================
# Step 6: Saving the cleaned dataset as a Delta Table
# ==========================================================
# Saving the cleaned dataset in Delta format.

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.superstore_delta")

print("Delta table created successfully.")


Delta table created successfully.


In [0]:
# Verifying the Changes Made in Col Names
delta_df = spark.table("workspace.default.superstore_delta")

display(delta_df.limit(20))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47
11,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.184,9,0.2,85.3092
12,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002033,Technology,Phones,Konftel 250 Conference�phone�- Charcoal black,911.424,4,0.2,68.3568
13,CA-2017-114412,2017-04-15,2017-04-20,Standard Class,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552,3,0.2,5.4432
14,CA-2016-161389,2016-12-05,2016-12-10,Standard Class,IM-15070,Irene Maddox,Consumer,United States,Seattle,Washington,98103,West,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,407.976,3,0.2,132.5922
15,US-2015-118983,2015-11-22,2015-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,Texas,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,"Holmes Replacement Filter for HEPA Air Cleaner, Very Large Room, HEPA Filter",68.81,5,0.8,-123.858


## Step 7: Creating an Incremental Dataset

To show incremental data, a second dataset is created from the original dataset.

The incremental dataset contains:
- Existing records with updated values (to demonstrate UPDATE).
- New records  (to demonstrate INSERT).

This shows how new data arrives in real-world data engineering pipelines.


In [0]:
display(clean_df.select("Order_ID").limit(10))

Order_ID
CA-2016-152156
CA-2016-152156
CA-2016-138688
US-2015-108966
US-2015-108966
CA-2014-115812
CA-2014-115812
CA-2014-115812
CA-2014-115812
CA-2014-115812


In [0]:
# ==========================================================
# Step 7: Creating Updated Records
# ==========================================================
# Select the first 5 records and increase the Sales value.
# These records will be updated during the MERGE operation.

updated_records = (
    clean_df
    .filter(col("Row_ID").isin(1, 2, 3, 4, 5))
    .withColumn("Sales", col("Sales") + 100)
)

display(updated_records)



Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,361.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",831.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,114.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1057.5774999999999,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,122.368,2,0.2,2.5164


In [0]:
# ==========================================================
# Step 7B: Creating  New Records
# ==========================================================
# Created a new records by assigning new Order IDs.

new_records = (
    updated_records
    .withColumn("Row_ID", col("Row_ID") + 100000)
)

display(new_records)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
100001,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,361.96,2,0.0,41.9136
100002,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",831.94,3,0.0,219.582
100003,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,114.62,2,0.0,6.8714
100004,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1057.5774999999999,5,0.45,-383.031
100005,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,122.368,2,0.2,2.5164


In [0]:
# ==========================================================
# Step 7C: Combining the  Updated and New Records
# ==========================================================
# Union both datasets to demonstrate incremental data.

incremental_df = updated_records.unionByName(new_records)

display(incremental_df)

print("Incremental Record Count:", incremental_df.count())

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,361.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",831.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,114.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1057.5774999999999,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,122.368,2,0.2,2.5164
100001,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,361.96,2,0.0,41.9136
100002,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",831.94,3,0.0,219.582
100003,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,114.62,2,0.0,6.8714
100004,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1057.5774999999999,5,0.45,-383.031
100005,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,122.368,2,0.2,2.5164


Incremental Record Count: 10


In [0]:
# verify no duplicates present before merge
incremental_df.groupBy("Row_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



## Step 8: Perform Delta Lake MERGE

In this step, the incremental dataset is merged with the Delta table.

The MERGE operation performs:
- UPDATE when the Order_ID already exists.
- INSERT when the Order_ID is new.



In [0]:
# ==========================================================
# Step 8: Load the Delta Table
# ==========================================================
# Load the Delta table created in the previous step.

delta_table = DeltaTable.forName(
    spark,
    "workspace.default.superstore_delta"
)
print("done")

done


In [0]:
# ==========================================================
# Step 8B: MERGING Incremental Data
# ==========================================================
# Updating matching records and insert new records.

(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE operation completed successfully.")

MERGE operation completed successfully.


## Step 9: Validate the MERGE Results

After performing the MERGE operation, the Delta table is validated to ensure:

- Existing records were updated successfully.
- New records were inserted successfully.
- No duplicate Row_ID values exist.
- Final row count is correct.

In [0]:
# ==========================================================
# Step 9: Displaying the Final Delta Table
# ==========================================================

final_df = spark.table("workspace.default.superstore_delta")

display(final_df.limit(20))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47
11,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.184,9,0.2,85.3092
12,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002033,Technology,Phones,Konftel 250 Conference�phone�- Charcoal black,911.424,4,0.2,68.3568
13,CA-2017-114412,2017-04-15,2017-04-20,Standard Class,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South,OFF-PA-10002365,Office Supplies,Paper,Xerox 1967,15.552,3,0.2,5.4432
14,CA-2016-161389,2016-12-05,2016-12-10,Standard Class,IM-15070,Irene Maddox,Consumer,United States,Seattle,Washington,98103,West,OFF-BI-10003656,Office Supplies,Binders,Fellowes PB200 Plastic Comb Binding Machine,407.976,3,0.2,132.5922
15,US-2015-118983,2015-11-22,2015-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,Texas,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,"Holmes Replacement Filter for HEPA Air Cleaner, Very Large Room, HEPA Filter",68.81,5,0.8,-123.858


In [0]:
# ==========================================================
# Step 9B: Checking the Final Row Count
# ==========================================================

print("Original Row Count :", clean_df.count())
print("Final Row Count    :", final_df.count())
print("New Rows Inserted  :", final_df.count() - clean_df.count())

Original Row Count : 9994
Final Row Count    : 9999
New Rows Inserted  : 5


In [0]:
# ==========================================================
# Step 9C: Checking Duplicate Row_ID
# ==========================================================

duplicate_rows = (
    final_df
    .groupBy("Row_ID")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_rows)

print("Duplicate Row_ID:", duplicate_rows.count())

Row_ID,count


Duplicate Row_ID: 0


In [0]:
# ==========================================================
# Step 9D: Verifying the Updated Records
# ==========================================================

display(
    final_df.filter(col("Row_ID").isin(1, 2, 3, 4, 5))
)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,361.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",831.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,114.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1057.5774999999999,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,122.368,2,0.2,2.5164


In [0]:
# ==========================================================
# Step 9E: Verifying Inserted Records
# ==========================================================

display(
    final_df.filter(col("Row_ID") >= 100001)
)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
100001,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,361.96,2,0.0,41.9136
100002,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",831.94,3,0.0,219.582
100003,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,114.62,2,0.0,6.8714
100004,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1057.5774999999999,5,0.45,-383.031
100005,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,122.368,2,0.2,2.5164


# Assignment Summary

### Objective

The objective of this assignment was to demonstrate incremental data processing using Delta Lake.

### Tasks Performed

- Loaded the Superstore dataset.
- Explored the dataset.
- Checked for missing values and duplicate records.
- Renamed columns to make them Delta-compatible.
- Created a Delta table.
- Generated an incremental dataset containing updates and new records.
- Applied the Delta Lake MERGE operation.
- Validated the merged data.

### Result

The Delta Lake MERGE operation successfully updated existing records and inserted new records into the Delta table. The final validation confirmed that the merge was successful and no duplicate Row_ID values were introduced.